# BERT / SciBERT Transfer Learning on Han Pairwise Dataset

This notebook is aligned with the uploaded In-domain BERT notebook. It keeps the same model list, tokenizer strategy, max length, learning rate, batch sizes, epochs, weight decay, early stopping, and best-model criterion. The main changes are the Han dataset input, 8:1:1 train/development/test split, saved Transfer Learning models, and gold-dataset evaluation.

## 1. Install and Import

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.10.0+cu128
True
NVIDIA L40S


In [2]:
import sys

!conda install -y -c conda-forge pyarrow
!{sys.executable} -m pip install transformers evaluate accelerate
!{sys.executable} -m pip install datasets --no-build-isolation

Retrieving notices: done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/pytorch

  added / updated specs:
    - pyarrow


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    aws-crt-cpp-0.40.1         |       h102d43b_3         407 KB  conda-forge
    aws-sdk-cpp-1.11.833       |       hc7390e0_9         3.2 MB  conda-forge
    azure-core-cpp-1.16.3      |       h206d751_0         341 KB  conda-forge
    azure-identity-cpp-1.13.3  |       h71f81a8_2         245 KB  conda-forge
    azure-storage-blobs-cpp-12.18.0|       h74b55db_1         576 KB  conda-forge
    azure-storage-common-cpp-12.14.0|       hf596fc9_1         156 KB  conda-forge
    azure-storage-files-datalake-cpp-12.16.0|       h1f05bef_1         301 KB  conda-forge
    gflags-2.3.1               |       h54a6638_0         128 KB  co

In [3]:
import pyarrow
import transformers
import datasets
import evaluate
import accelerate

print("pyarrow:", pyarrow.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("evaluate:", evaluate.__version__)
print("accelerate:", accelerate.__version__)

pyarrow: 25.0.0
transformers: 5.15.1
datasets: 5.0.1
evaluate: 0.4.6
accelerate: 1.14.0


In [4]:
import os
import random
import json
import shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    classification_report,
    accuracy_score,
    roc_auc_score
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset

In [5]:
!pip install "huggingface_hub==0.23.4"

  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.1 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 0.23.4 which is incompatible.
transformers 5.15.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.23.4 which is incompatible.


In [6]:
import transformers
import huggingface_hub
import tokenizers

print(transformers.__version__)
print(huggingface_hub.__version__)
print(tokenizers.__version__)

5.15.1
1.28.0
0.22.2


In [7]:
from huggingface_hub import snapshot_download

local_scibert_dir = snapshot_download(
    repo_id="allenai/scibert_scivocab_uncased",
    local_dir="./models/scibert_scivocab_uncased",
    local_dir_use_symlinks=False,
    resume_download=True
)

print(local_scibert_dir)

/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  is thrown and the `use_auth_token` value is ignored.
/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  This has been discussed in:


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/home/ec2-user/SageMaker/Experiment/models/scibert_scivocab_uncased


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

local_scibert_dir = "./models/scibert_scivocab_uncased"

tokenizer = AutoTokenizer.from_pretrained(local_scibert_dir)

model = AutoModelForSequenceClassification.from_pretrained(
    local_scibert_dir,
    num_labels=2
)

print("SciBERT loaded from local directory.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ./models/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

SciBERT loaded from local directory.


## 2. Set random seed for reproducibility

In [9]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 3. Define paths, model list, and training configuration

In [10]:
# ==============================
# Configuration
# ==============================

DATA_PATH = "han_pairwise_dataset.csv"
GOLD_PATH = "pairwise_gold_dataset.csv"
OUTPUT_DIR = "bert_transfer_outputs"
MODEL_SAVE_DIR = os.path.join(OUTPUT_DIR, "saved_models")
PER_MODEL_RESULTS_DIR = os.path.join(OUTPUT_DIR, "per_model_results")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(PER_MODEL_RESULTS_DIR, exist_ok=True)

# Use 0.10 for a 10% pilot run. Use 1.00 for the full Han dataset.
SAMPLE_FRAC = 1.0

# Keep these aligned with the In-domain BERT notebook.
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 32
NUM_TRAIN_EPOCHS = 5
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 2

MODEL_LIST = {
    "BERT": "bert-base-uncased",
    "SciBERT": "./models/scibert_scivocab_uncased"
}

# Hardware target for this run (configured in SageMaker, not in Python)
EXPECTED_GPU_KEYWORD = "L40S"  # ml.g6e.4xlarge uses NVIDIA L40S


In [11]:
# ==============================
# Hardware verification
# Target SageMaker Notebook instance: ml.g6e.4xlarge (NVIDIA L40S)
# Note: the instance type itself must be changed in the SageMaker console.
# This cell only verifies what GPU the notebook kernel actually sees.
# ==============================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available. Check the SageMaker instance/kernel configuration.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU: {gpu_name}")
print(f"GPU memory: {gpu_mem_gb:.1f} GB")

if EXPECTED_GPU_KEYWORD.lower() in gpu_name.lower():
    print("Hardware check: OK — expected L40S GPU detected for ml.g6e.4xlarge.")
else:
    print(
        f"WARNING: Expected an L40S GPU for ml.g6e.4xlarge, but detected: {gpu_name}. "
        "The experiment can still run, but verify the SageMaker instance type before the full run."
    )


GPU: NVIDIA L40S
GPU memory: 44.4 GB
Hardware check: OK — expected L40S GPU detected for ml.g6e.4xlarge.


## 4. Load Han pairwise dataset

In [12]:
# Load Han pairwise dataset
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

required_cols = ["OrgName_1", "OrgName_2", "Label"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].dropna().copy().reset_index(drop=True)
df["OrgName_1"] = df["OrgName_1"].astype(str)
df["OrgName_2"] = df["OrgName_2"].astype(str)
df["Label"] = df["Label"].astype(int)

print("Full dataset shape:", df.shape)
print(df["Label"].value_counts())

if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Using SAMPLE_FRAC={SAMPLE_FRAC}")
    print("Sampled dataset shape:", df.shape)
    print(df["Label"].value_counts())

Full dataset shape: (905357, 3)
Label
0    715157
1    190200
Name: count, dtype: int64


## 5. Split data into Train / Development / Test sets

In [13]:
# Train/Development/Test split = 8:1:1
# Development is used for early stopping, as in the In-domain BERT notebook.

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=df["Label"]
)

dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_df["Label"]
)

train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape, train_df["Label"].value_counts().to_dict())
print("Dev:", dev_df.shape, dev_df["Label"].value_counts().to_dict())
print("Test:", test_df.shape, test_df["Label"].value_counts().to_dict())

train_df.to_csv(os.path.join(OUTPUT_DIR, "han_train_split.csv"), index=False, encoding="utf-8-sig")
dev_df.to_csv(os.path.join(OUTPUT_DIR, "han_dev_split.csv"), index=False, encoding="utf-8-sig")
test_df.to_csv(os.path.join(OUTPUT_DIR, "han_test_split.csv"), index=False, encoding="utf-8-sig")

Train: (724285, 3) {0: 572125, 1: 152160}
Dev: (90536, 3) {0: 71516, 1: 19020}
Test: (90536, 3) {0: 71516, 1: 19020}


## 6. Convert pandas DataFrame to Hugging Face Dataset

In [14]:
def make_hf_dataset(dataframe):
    # Keep only columns needed for model training
    temp_df = dataframe[["OrgName_1", "OrgName_2", "Label"]].copy()
    
    # Rename Label to labels for Hugging Face Trainer
    temp_df = temp_df.rename(columns={"Label": "labels"})
    
    return Dataset.from_pandas(temp_df, preserve_index=False)

## 7. Tokenization function

In [15]:
def tokenize_dataset(dataset, tokenizer, max_length=MAX_LENGTH):
    # Tokenize organization name pairs as sentence pairs
    # This is intentionally aligned with the In-domain BERT notebook.
    def tokenize_batch(batch):
        return tokenizer(
            batch["OrgName_1"],
            batch["OrgName_2"],
            padding="max_length",
            truncation=True,
            max_length=max_length
        )
    
    tokenized = dataset.map(tokenize_batch, batched=True)
    
    # Set PyTorch format
    tokenized = tokenized.remove_columns(["OrgName_1", "OrgName_2"])
    tokenized.set_format("torch")
    
    return tokenized

## 8. Metric function for Trainer

In [16]:
def compute_metrics(eval_pred):
    # Compute positive-class and negative-class precision, recall, and F1
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1],
        average=None,
        zero_division=0
    )
    
    return {
        "nonmatch_precision": class_precision[0],
        "nonmatch_recall": class_recall[0],
        "nonmatch_f1": class_f1[0],
        "match_precision": class_precision[1],
        "match_recall": class_recall[1],
        "match_f1": class_f1[1],
    }

## 9. Evaluation helper for Match / NonMatch summaries

In [17]:
def evaluate_predictions(y_true, y_pred, y_prob_match, model_label, dataset_name):
    class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        average=None,
        zero_division=0
    )
    
    accuracy = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob_match) if len(np.unique(y_true)) == 2 else np.nan
    
    summary_rows = [
        {
            "Dataset": dataset_name,
            "Model": model_label,
            "Class": "NonMatch",
            "Precision": class_precision[0],
            "Recall": class_recall[0],
            "F1": class_f1[0],
            "Support": class_support[0],
            "Accuracy": accuracy,
            "ROC_AUC": auc
        },
        {
            "Dataset": dataset_name,
            "Model": model_label,
            "Class": "Match",
            "Precision": class_precision[1],
            "Recall": class_recall[1],
            "F1": class_f1[1],
            "Support": class_support[1],
            "Accuracy": accuracy,
            "ROC_AUC": auc
        }
    ]
    
    print(f"===== {dataset_name} evaluation: {model_label} =====")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["NonMatch", "Match"],
        digits=4,
        zero_division=0
    ))
    print("Accuracy:", accuracy)
    if len(np.unique(y_true)) == 2:
        print("ROC_AUC:", auc)
    
    return pd.DataFrame(summary_rows)

## 10. Main training and Han test evaluation function

### Restart-safe execution (BERT → SciBERT)

Each model now saves its **best/final model, tokenizer, configuration, Han-test predictions, and Han-test summary immediately after that model finishes**. A `COMPLETE.json` marker is created only after every required artifact has been written successfully.

On rerun, a model with a valid completion marker is skipped and its saved results are loaded from disk. A model without `COMPLETE.json` is treated as incomplete and is restarted from the beginning. This changes only experiment orchestration/storage; the original learning rate, batch sizes, epoch limit, early stopping, random seed, and model definitions are unchanged.


In [18]:
# ==============================
# Restart-safe training helpers
# ==============================

def atomic_to_csv(dataframe, path):
    """Write CSV via a temporary file, then atomically replace the destination."""
    tmp_path = path + ".tmp"
    dataframe.to_csv(tmp_path, index=False, encoding="utf-8-sig")
    os.replace(tmp_path, path)


def get_model_run_paths(model_label):
    result_dir = os.path.join(PER_MODEL_RESULTS_DIR, model_label)
    os.makedirs(result_dir, exist_ok=True)
    return {
        "result_dir": result_dir,
        "prediction_path": os.path.join(result_dir, f"{model_label}_han_test_predictions_with_probability.csv"),
        "summary_path": os.path.join(result_dir, f"{model_label}_han_test_summary_match_nonmatch.csv"),
        "complete_flag": os.path.join(result_dir, "COMPLETE.json"),
        "saved_model_dir": os.path.join(MODEL_SAVE_DIR, f"{model_label}_han_transfer_model"),
        "trainer_output_dir": os.path.join(OUTPUT_DIR, f"outputs_{model_label}_han_transfer"),
    }


def model_run_is_complete(model_label):
    paths = get_model_run_paths(model_label)
    required = [
        paths["complete_flag"],
        paths["prediction_path"],
        paths["summary_path"],
        paths["saved_model_dir"],
    ]
    return all(os.path.exists(p) for p in required)


def load_completed_model_results(model_label):
    paths = get_model_run_paths(model_label)
    summary_df = pd.read_csv(paths["summary_path"], encoding="utf-8-sig")
    pred_df = pd.read_csv(paths["prediction_path"], encoding="utf-8-sig")
    print(f"{model_label}: COMPLETE found — skipping training and loading saved results.")
    return summary_df, pred_df, paths["saved_model_dir"]


def prepare_fresh_model_run(model_label):
    """Remove incomplete artifacts so an interrupted model restarts cleanly from the beginning."""
    paths = get_model_run_paths(model_label)

    if os.path.exists(paths["complete_flag"]):
        os.remove(paths["complete_flag"])

    for file_path in [paths["prediction_path"], paths["summary_path"]]:
        if os.path.exists(file_path):
            os.remove(file_path)

    # If a previous run stopped before COMPLETE was written, do not trust partial model/checkpoint files.
    for dir_path in [paths["saved_model_dir"], paths["trainer_output_dir"]]:
        if os.path.isdir(dir_path):
            print(f"Removing incomplete artifacts: {dir_path}")
            shutil.rmtree(dir_path)

    return paths


def train_and_evaluate_transformer(train_df, dev_df, test_df, model_label, model_name):
    print("==============================")
    print(f"Training {model_label}")
    print("==============================")
    print("Train:", train_df.shape, train_df["Label"].value_counts().to_dict())
    print("Dev:", dev_df.shape, dev_df["Label"].value_counts().to_dict())
    print("Test:", test_df.shape, test_df["Label"].value_counts().to_dict())

    # A model without COMPLETE is intentionally restarted from scratch.
    paths = prepare_fresh_model_run(model_label)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    train_dataset = make_hf_dataset(train_df)
    dev_dataset = make_hf_dataset(dev_df)
    test_dataset = make_hf_dataset(test_df)

    train_tokenized = tokenize_dataset(train_dataset, tokenizer)
    dev_tokenized = tokenize_dataset(dev_dataset, tokenizer)
    test_tokenized = tokenize_dataset(test_dataset, tokenizer)

    # Training arguments are unchanged from the original notebook.
    # Hardware is upgraded separately to ml.g6e.4xlarge; no batch-size, LR, epoch,
    # precision, or early-stopping setting is changed here.
    training_args = TrainingArguments(
        output_dir=paths["trainer_output_dir"],
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_match_f1",
        greater_is_better=True,
        save_total_limit=1,
        seed=RANDOM_SEED,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=dev_tokenized,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)
        ]
    )

    trainer.train()

    # 1) Save final/best model and tokenizer immediately.
    trainer.save_model(paths["saved_model_dir"])
    tokenizer.save_pretrained(paths["saved_model_dir"])

    config = {
        "model_label": model_label,
        "model_name": model_name,
        "max_length": MAX_LENGTH,
        "learning_rate": LEARNING_RATE,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "random_seed": RANDOM_SEED,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    }
    with open(os.path.join(paths["saved_model_dir"], "transfer_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    # 2) Predict on Han test set.
    test_output = trainer.predict(test_tokenized)
    logits = test_output.predictions
    y_test = test_df["Label"].values

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    y_prob_match = probs[:, 1]
    y_prob_nonmatch = probs[:, 0]
    y_pred = np.argmax(probs, axis=1)

    summary_df = evaluate_predictions(
        y_true=y_test,
        y_pred=y_pred,
        y_prob_match=y_prob_match,
        model_label=model_label,
        dataset_name="Han_Test"
    )

    pred_df = test_df.copy()
    pred_df["Model"] = model_label
    pred_df["TrueLabel"] = y_test
    pred_df["PredictedLabel"] = y_pred
    pred_df["Probability_Match"] = y_prob_match
    pred_df["Probability_NonMatch"] = y_prob_nonmatch

    # 3) Save this model's results immediately.
    atomic_to_csv(pred_df, paths["prediction_path"])
    atomic_to_csv(summary_df, paths["summary_path"])

    # 4) COMPLETE is written LAST. Its presence means model + tokenizer + config +
    #    predictions + summary were all successfully written.
    complete_metadata = {
        "status": "COMPLETE",
        "model_label": model_label,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "saved_model_dir": paths["saved_model_dir"],
        "prediction_path": paths["prediction_path"],
        "summary_path": paths["summary_path"],
    }
    complete_tmp = paths["complete_flag"] + ".tmp"
    with open(complete_tmp, "w", encoding="utf-8") as f:
        json.dump(complete_metadata, f, indent=2)
    os.replace(complete_tmp, paths["complete_flag"])

    print(f"{model_label}: model/results saved and COMPLETE marker written.")

    del model
    del trainer
    torch.cuda.empty_cache()

    return summary_df, pred_df, paths["saved_model_dir"]


## 11. Run BERT and SciBERT Transfer Learning

In [ ]:
# ==============================
# Restart-safe BERT -> SciBERT execution
# ==============================
# Behavior on rerun:
#   - COMPLETE exists: load that model's saved CSVs and skip training.
#   - COMPLETE missing: restart only that model from the beginning.
# Thus, if BERT finished before a browser/session interruption, SciBERT can resume next.

all_summary = []
all_predictions = []
saved_model_dirs = {}

for model_label, model_name in MODEL_LIST.items():
    if model_run_is_complete(model_label):
        summary_df, pred_df, saved_model_dir = load_completed_model_results(model_label)
    else:
        summary_df, pred_df, saved_model_dir = train_and_evaluate_transformer(
            train_df=train_df,
            dev_df=dev_df,
            test_df=test_df,
            model_label=model_label,
            model_name=model_name
        )

    all_summary.append(summary_df)
    all_predictions.append(pred_df)
    saved_model_dirs[model_label] = saved_model_dir

transformer_summary_df = pd.concat(all_summary, ignore_index=True)
transformer_pair_predictions_df = pd.concat(all_predictions, ignore_index=True)

print("Saved model directories:")
print(saved_model_dirs)


Training BERT
Train: (724285, 3) {0: 572125, 1: 152160}
Dev: (90536, 3) {0: 71516, 1: 19020}
Test: (90536, 3) {0: 71516, 1: 19020}


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

... (24 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

===== Han_Test evaluation: SciBERT =====
              precision    recall  f1-score   support

    NonMatch     0.9754    0.9834    0.9794     71516
       Match     0.9356    0.9067    0.9209     19020

    accuracy                         0.9673     90536
   macro avg     0.9555    0.9450    0.9501     90536
weighted avg     0.9670    0.9673    0.9671     90536

Accuracy: 0.9672726871078908
ROC_AUC: 0.9884475191009736
SciBERT: model/results saved and COMPLETE marker written.
Saved model directories:
{'BERT': 'bert_transfer_outputs/saved_models/BERT_han_transfer_model', 'SciBERT': 'bert_transfer_outputs/saved_models/SciBERT_han_transfer_model'}


## 12. Save Han test results

In [ ]:
# ==============================
# Rebuild/save the same final combined Han test files as the original notebook
# ==============================
# Because transformer_* dataframes are rebuilt from each model's saved per-model CSVs
# when necessary, this cell is safe to rerun after a session/browser interruption.

transformer_pair_predictions_df.to_csv(
    os.path.join(OUTPUT_DIR, "bert_scibert_han_pair_level_predictions_with_probability.csv"),
    index=False,
    encoding="utf-8-sig"
)

transformer_summary_df.to_csv(
    os.path.join(OUTPUT_DIR, "bert_scibert_han_summary_match_nonmatch.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("Final combined Han test files saved.")
transformer_summary_df


Final combined Han test files saved.


,Dataset,Model,Class,Precision,Recall,F1,Support,Accuracy,ROC_AUC
0,Han_Test,BERT,NonMatch,0.977242,0.986493,0.981845,71516,0.971183,0.990735
1,Han_Test,BERT,Match,0.947337,0.913617,0.930172,19020,0.971183,0.990735
2,Han_Test,SciBERT,NonMatch,0.975382,0.983388,0.979369,71516,0.967273,0.988448
3,Han_Test,SciBERT,Match,0.935550,0.906677,0.920888,19020,0.967273,0.988448


## 13. Function for applying saved BERT/SciBERT models to pairwise_gold_dataset.csv

In [ ]:
def load_gold_dataset(gold_path):
    gold_df = pd.read_csv(gold_path, encoding="utf-8-sig")
    required_cols = ["OrgName_1", "OrgName_2", "Label"]
    missing_cols = [col for col in required_cols if col not in gold_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in gold dataset: {missing_cols}")
    
    gold_df = gold_df[required_cols].dropna().copy().reset_index(drop=True)
    gold_df["OrgName_1"] = gold_df["OrgName_1"].astype(str)
    gold_df["OrgName_2"] = gold_df["OrgName_2"].astype(str)
    gold_df["Label"] = gold_df["Label"].astype(int)
    return gold_df


def evaluate_saved_transformer_on_gold(
    gold_path,
    saved_model_dir,
    model_label,
    output_prediction_path=None,
    output_summary_path=None
):
    gold_df = load_gold_dataset(gold_path)
    
    tokenizer = AutoTokenizer.from_pretrained(saved_model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(saved_model_dir)
    
    gold_dataset = make_hf_dataset(gold_df)
    gold_tokenized = tokenize_dataset(gold_dataset, tokenizer)
    
    training_args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, f"gold_eval_{model_label}"),
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args
    )
    
    gold_output = trainer.predict(gold_tokenized)
    logits = gold_output.predictions
    y_true = gold_df["Label"].values
    
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    y_prob_match = probs[:, 1]
    y_prob_nonmatch = probs[:, 0]
    y_pred = np.argmax(probs, axis=1)
    
    summary_df = evaluate_predictions(
        y_true=y_true,
        y_pred=y_pred,
        y_prob_match=y_prob_match,
        model_label=model_label,
        dataset_name="Gold"
    )
    
    pred_df = gold_df.copy()
    pred_df["Model"] = model_label
    pred_df["TrueLabel"] = y_true
    pred_df["PredictedLabel"] = y_pred
    pred_df["Probability_Match"] = y_prob_match
    pred_df["Probability_NonMatch"] = y_prob_nonmatch
    
    if output_prediction_path is not None:
        pred_df.to_csv(output_prediction_path, index=False, encoding="utf-8-sig")
        print("Saved predictions to:", output_prediction_path)
    
    if output_summary_path is not None:
        summary_df.to_csv(output_summary_path, index=False, encoding="utf-8-sig")
        print("Saved summary to:", output_summary_path)
    
    del model
    del trainer
    torch.cuda.empty_cache()
    
    return pred_df, summary_df

## 14. Evaluate saved Han-trained BERT/SciBERT models on the gold dataset

In [ ]:
# Evaluate BERT and SciBERT on pairwise_gold_dataset.csv
bert_gold_pred, bert_gold_summary = evaluate_saved_transformer_on_gold(
    gold_path=GOLD_PATH,
    saved_model_dir=os.path.join(MODEL_SAVE_DIR, "BERT_han_transfer_model"),
    model_label="BERT",
    output_prediction_path=os.path.join(OUTPUT_DIR, "bert_gold_predictions_with_probability.csv"),
    output_summary_path=os.path.join(OUTPUT_DIR, "bert_gold_summary_match_nonmatch.csv")
)

scibert_gold_pred, scibert_gold_summary = evaluate_saved_transformer_on_gold(
    gold_path=GOLD_PATH,
    saved_model_dir=os.path.join(MODEL_SAVE_DIR, "SciBERT_han_transfer_model"),
    model_label="SciBERT",
    output_prediction_path=os.path.join(OUTPUT_DIR, "scibert_gold_predictions_with_probability.csv"),
    output_summary_path=os.path.join(OUTPUT_DIR, "scibert_gold_summary_match_nonmatch.csv")
)

bert_scibert_gold_summary = pd.concat(
    [bert_gold_summary, scibert_gold_summary],
    ignore_index=True
)

bert_scibert_gold_predictions = pd.concat(
    [bert_gold_pred, scibert_gold_pred],
    ignore_index=True
)

bert_scibert_gold_summary.to_csv(
    os.path.join(OUTPUT_DIR, "bert_scibert_gold_summary_match_nonmatch.csv"),
    index=False,
    encoding="utf-8-sig"
)

bert_scibert_gold_predictions.to_csv(
    os.path.join(OUTPUT_DIR, "bert_scibert_gold_pair_level_predictions_with_probability.csv"),
    index=False,
    encoding="utf-8-sig"
)

bert_scibert_gold_summary

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/3396 [00:00<?, ? examples/s]

===== Gold evaluation: BERT =====
              precision    recall  f1-score   support

    NonMatch     0.9313    0.6848    0.7892      2830
       Match     0.3217    0.7473    0.4498       566

    accuracy                         0.6952      3396
   macro avg     0.6265    0.7161    0.6195      3396
weighted avg     0.8297    0.6952    0.7327      3396

Accuracy: 0.6952296819787986
ROC_AUC: 0.7825874964102436
Saved predictions to: bert_transfer_outputs/bert_gold_predictions_with_probability.csv
Saved summary to: bert_transfer_outputs/bert_gold_summary_match_nonmatch.csv


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/3396 [00:00<?, ? examples/s]

===== Gold evaluation: SciBERT =====
              precision    recall  f1-score   support

    NonMatch     0.9265    0.6728    0.7795      2830
       Match     0.3095    0.7332    0.4352       566

    accuracy                         0.6829      3396
   macro avg     0.6180    0.7030    0.6074      3396
weighted avg     0.8237    0.6829    0.7221      3396

Accuracy: 0.6828621908127208
ROC_AUC: 0.7486318345840253
Saved predictions to: bert_transfer_outputs/scibert_gold_predictions_with_probability.csv
Saved summary to: bert_transfer_outputs/scibert_gold_summary_match_nonmatch.csv


,Dataset,Model,Class,Precision,Recall,F1,Support,Accuracy,ROC_AUC
0,Gold,BERT,NonMatch,0.931283,0.684806,0.789249,2830,0.695230,0.782587
1,Gold,BERT,Match,0.321673,0.747350,0.449761,566,0.695230,0.782587
2,Gold,SciBERT,NonMatch,0.926521,0.672792,0.779529,2830,0.682862,0.748632
3,Gold,SciBERT,Match,0.309471,0.733216,0.435239,566,0.682862,0.748632


## 15. Alignment with the In-domain BERT notebook

The following settings are intentionally kept the same as the uploaded In-domain notebook:

- Models: `bert-base-uncased`, `allenai/scibert_scivocab_uncased`
- Tokenization: `tokenizer(OrgName_1, OrgName_2, padding="max_length", truncation=True, max_length=128)`
- Learning rate: `2e-5`
- Train batch size: `16`
- Eval batch size: `32`
- Epochs: `5`
- Weight decay: `0.01`
- Early stopping patience: `2`
- Best-model metric: `eval_match_f1`
- Seed: `42`

The main methodological difference is the data design: In-domain Learning uses 10-bin cross-validation on the gold dataset, while Transfer Learning trains on the Han dataset using an 8:1:1 train/dev/test split and then evaluates the saved model on the gold dataset.